**In-Class Exercise:** Heterogeneous Treatment Effects & Causal Trees

**Course:** ECON6083 - Machine Learning in Economics
**Topic:** EMSE Criterion, Honest Estimation, DML, Causal Forests, GRF with IV

---

## Recap: 401(k) DML (from Lecture 5)

Before moving to heterogeneous effects, let's replicate the 401(k) ATE result from last lecture using `DoubleML`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, KFold

import doubleml as dml
from doubleml.datasets import fetch_401K

data_401k = fetch_401K()
print(f"Sample size: {data_401k.n_obs}")
print(f"Number of covariates: {data_401k.x.shape[1]}")

ml_l = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
ml_m = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

dml_401k = dml.DoubleMLPLR(data_401k, ml_l, ml_m, n_folds=3)
dml_401k.fit()

theta = dml_401k.coef[0]
ci = dml_401k.confint()

print(f"\nDML with Random Forest:")
print(f"  Estimate: ${theta:,.2f}")
print(f"  95% CI:   [${ci.iloc[0, 0]:,.2f}, ${ci.iloc[0, 1]:,.2f}]")
print("Note: Chernozhukov et al. (2018) finds ~$8,000-$9,000")

## Part A: Causal Tree Pitfalls

Now we move beyond ATE → estimating **heterogeneous** treatment effects $\tau(x)$.

### Setup: Generate Data with Heterogeneous Treatment Effects

In [ ]:
np.random.seed(42)
n = 2000

X = np.random.normal(0, 1, (n, 2))
X1, X2 = X[:, 0], X[:, 1]

# True CATE: tau(x) = 2 if X1 > 0, else 0.5
true_tau = np.where(X1 > 0, 2.0, 0.5)

# Propensity score e(x) depends on X1 (confounding)
e_true = 1 / (1 + np.exp(-0.5 * X1))
W = (np.random.uniform(0, 1, n) < e_true).astype(int)

# Potential outcomes and observed Y
Y0 = 1 + 2 * X2 + np.random.normal(0, 0.5, n)
Y1 = Y0 + true_tau
Y = W * Y1 + (1 - W) * Y0

print(f"Data: N={n}")
print(f"True CATE: tau = 2.0 if X1 > 0, tau = 0.5 if X1 <= 0")
print(f"True ATE: {np.mean(true_tau):.2f}")

### A.1: Standard CART — Wrong Target

What happens if we just fit a standard regression tree on $Y$?

In [ ]:
tree_Y = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_Y.fit(X, Y)

print("Standard tree (predicting Y, not tau):")
print(export_text(tree_Y, feature_names=['X1', 'X2'])[:500])
print("\nProblem: Splits on X2 (predicts Y) instead of X1 (moderates tau).")
print("Standard CART minimizes MSE(Y), not EMSE(tau).")

### A.2: Splitting Criterion — Why $\hat{\tau}^2$ Alone is Wrong

The naive criterion maximizes $\hat{\tau}^2$ across leaves. But:

$$E[\hat{\tau}^2] = \tau^2 + \text{Var}(\hat{\tau})$$

So $\hat{\tau}^2$ is **upward biased**. The correct criterion (EMSE) subtracts the variance:

$$\text{EMSE} = \hat{\tau}^2 - \hat{V}(\hat{\tau})$$

Let's verify this with Monte Carlo.

In [ ]:
# Monte Carlo: show E[tau_hat^2] > tau^2
n_mc = 500
n_demo = 400
tau_true_demo = 1.0  # constant effect, NO heterogeneity

tau_sq_accum = []
emse_accum = []

for mc in range(n_mc):
    rng = np.random.RandomState(mc)
    X_d = rng.normal(0, 1, n_demo)
    W_d = rng.binomial(1, 0.5, n_demo)
    Y_d = 0.5 * X_d + tau_true_demo * W_d + rng.normal(0, 1, n_demo)

    med = np.median(X_d)
    for mask in [X_d <= med, X_d > med]:
        Wl, Yl = W_d[mask], Y_d[mask]
        nt, nc = np.sum(Wl == 1), np.sum(Wl == 0)
        if nt < 3 or nc < 3:
            continue

        tau_hat = np.mean(Yl[Wl == 1]) - np.mean(Yl[Wl == 0])  # <-- DIM estimator
        V_hat = (np.var(Yl[Wl == 1], ddof=1) / nt +             # <-- Variance of DIM
                 np.var(Yl[Wl == 0], ddof=1) / nc)

        tau_sq_accum.append(tau_hat ** 2)          # naive criterion
        emse_accum.append(tau_hat ** 2 - V_hat)    # EMSE = debiased

print(f"Monte Carlo results ({n_mc} replications, true tau = {tau_true_demo}):")
print(f"  True tau^2          = {tau_true_demo ** 2:.4f}")
print(f"  Mean tau_hat^2      = {np.mean(tau_sq_accum):.4f}  (biased ABOVE)")
print(f"  Mean EMSE           = {np.mean(emse_accum):.4f}  (debiased)")

### A.3: Honest Estimation (same idea as DML cross-fitting from Lecture 5)

Split the sample: learn tree structure on $S_{tr}$, estimate effects on $S_{est}$.

**Task**: Fill in the blanks.

In [ ]:
# Step 1: Split sample (analogous to DML cross-fitting)
X_struct, X_est, W_struct, W_est, Y_struct, Y_est = train_test_split(
    X, W, Y, test_size=0.5, random_state=42)       # <-- why 0.5?

# Step 2: Learn tree structure on S_tr using transformed outcome
e_hat = 0.5  # simplified; in practice estimate propensity
Y_trans = Y_struct * (2 * W_struct - 1) / e_hat     # <-- IPW transform
struct_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
struct_tree.fit(X_struct, Y_trans)                   # <-- fit on S_tr only!

# Step 3: Estimate tau in each leaf using S_est (NEVER S_tr!)
leaves = struct_tree.apply(X_est)                    # <-- apply to S_est!
print("Honest leaf-level treatment effects:")
for leaf in np.unique(leaves):
    mask = leaves == leaf
    Wl, Yl = W_est[mask], Y_est[mask]
    nt, nc = np.sum(Wl == 1), np.sum(Wl == 0)
    if nt >= 3 and nc >= 3:
        tau_hat = np.mean(Yl[Wl == 1]) - np.mean(Yl[Wl == 0])  # <-- DIM
        se = np.sqrt(np.var(Yl[Wl == 1], ddof=1)/nt +
                     np.var(Yl[Wl == 0], ddof=1)/nc)
        print(f"  Leaf {leaf}: tau_hat = {tau_hat:.3f}, SE = {se:.3f}, n = {np.sum(mask)}")

## Part B: DML — Plug-in vs Orthogonal Nuisance

| Method | Nuisance Rate | Target Rate | Inference |
|:---|:---:|:---:|:---:|
| Plug-in ML | $n^{-1/4}$ | $n^{-1/4}$ | Invalid |
| DML (orthogonal) | $n^{-1/4}$ | $n^{-1/2}$ | Valid |

### B.1: Setup — Confounding with Nonlinear Nuisance

In [ ]:
np.random.seed(42)
n = 3000

X_dml = np.random.normal(0, 1, (n, 5))
m_true = 2 * X_dml[:, 0] + X_dml[:, 1] ** 2 + np.sin(X_dml[:, 2])
e_true_dml = 1 / (1 + np.exp(-(0.5 * X_dml[:, 0] + 0.3 * X_dml[:, 1])))

W_dml = (np.random.uniform(0, 1, n) < e_true_dml).astype(int)
tau_true_dml = 2.0
Y_dml = m_true + tau_true_dml * W_dml + np.random.normal(0, 1, n)

print(f"True ATE: tau = {tau_true_dml}")
print(f"Nuisance: m(x) = 2*X1 + X2^2 + sin(X3)")

### B.2: Plug-in Nuisance (Non-orthogonal) — Biased

Estimate nuisance on FULL sample (no cross-fitting), plug in directly.

In [ ]:
# Estimate nuisance on FULL sample (no cross-fitting)
m_hat_plugin = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
m_hat_plugin.fit(X_dml, Y_dml)
m_pred_plugin = m_hat_plugin.predict(X_dml)

e_hat_plugin = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
e_hat_plugin.fit(X_dml, W_dml)
e_pred_plugin = np.clip(e_hat_plugin.predict_proba(X_dml)[:, 1], 0.01, 0.99)

# Plug-in IPW
tau_plugin_ipw = np.mean(
    Y_dml * W_dml / e_pred_plugin - Y_dml * (1 - W_dml) / (1 - e_pred_plugin))

# Plug-in regression
W_resid_plugin = W_dml - e_pred_plugin
Y_resid_plugin = Y_dml - m_pred_plugin
tau_plugin_reg = np.sum(Y_resid_plugin * W_resid_plugin) / np.sum(W_resid_plugin ** 2)

print(f"Plug-in IPW:        tau_hat = {tau_plugin_ipw:.4f}  (bias = {tau_plugin_ipw - tau_true_dml:+.4f})")
print(f"Plug-in regression: tau_hat = {tau_plugin_reg:.4f}  (bias = {tau_plugin_reg - tau_true_dml:+.4f})")

### B.3: DML (Orthogonal + Cross-fitting) — Valid Inference

**Task**: Fill in the cross-fitting loop.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
Y_residual = np.zeros(n)
W_residual = np.zeros(n)

for train_idx, test_idx in kf.split(X_dml):
    # Train nuisance on training fold
    m_model = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
    m_model.fit(X_dml[train_idx], Y_dml[train_idx])

    e_model = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
    e_model.fit(X_dml[train_idx], W_dml[train_idx])

    # Predict on HELD-OUT fold                          <-- key: cross-fitting!
    Y_residual[test_idx] = Y_dml[test_idx] - m_model.predict(X_dml[test_idx])
    e_pred = np.clip(e_model.predict_proba(X_dml[test_idx])[:, 1], 0.01, 0.99)
    W_residual[test_idx] = W_dml[test_idx] - e_pred

# DML estimator: orthogonal moment
tau_dml = np.sum(Y_residual * W_residual) / np.sum(W_residual ** 2)

# Standard error
psi = W_residual * (Y_residual - tau_dml * W_residual)
J = np.mean(W_residual ** 2)
se_dml = np.sqrt(np.mean(psi ** 2)) / (J * np.sqrt(n))

print(f"DML estimate:  tau_hat = {tau_dml:.4f}  (bias = {tau_dml - tau_true_dml:+.4f})")
print(f"95% CI: [{tau_dml - 1.96 * se_dml:.4f}, {tau_dml + 1.96 * se_dml:.4f}]")
print(f"Covers true tau? {'Yes' if abs(tau_dml - tau_true_dml) < 1.96 * se_dml else 'No'}")

### B.4: Summary Comparison

In [ ]:
print(f"{'Method':<25} {'tau_hat':>10} {'Bias':>10}")
print("-" * 47)
print(f"{'Plug-in IPW':<25} {tau_plugin_ipw:10.4f} {tau_plugin_ipw - tau_true_dml:+10.4f}")
print(f"{'Plug-in Regression':<25} {tau_plugin_reg:10.4f} {tau_plugin_reg - tau_true_dml:+10.4f}")
print(f"{'DML (orthogonal)':<25} {tau_dml:10.4f} {tau_dml - tau_true_dml:+10.4f}")
print(f"{'True tau':<25} {tau_true_dml:10.4f} {0:+10.4f}")

## Part C: Causal Forests in Practice

### C.1: CausalForestDML (handles confounding via DML residualization)

In [ ]:
np.random.seed(42)
n = 2000

age = np.random.normal(45, 10, n)
income = np.random.normal(50000, 20000, n)
X_cf = pd.DataFrame({'age': age, 'income': income / 10000})

W_cf = (np.random.uniform(0, 1, n) < 1 / (1 + np.exp(-(age - 45) / 10))).astype(int)
true_tau_cf = 2000 + 0.3 * (income - 50000) + 50 * (age - 45)
Y_cf = 5000 + true_tau_cf * W_cf + 500 * age + np.random.normal(0, 2000, n)

X_train, X_test, Y_train, Y_test, W_train, W_test, idx_train, idx_test = \
    train_test_split(X_cf, Y_cf, W_cf, np.arange(n), test_size=0.3, random_state=42)
true_test = true_tau_cf[idx_test]

print(f"Training: n={len(X_train)}, Test: n={len(X_test)}")
print(f"True ATE: ${true_tau_cf.mean():.0f}")

In [ ]:
from econml.dml import CausalForestDML             # <-- key API

est = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=50, random_state=42),
    model_t=RandomForestRegressor(n_estimators=50, random_state=42),
    n_estimators=1000,
    min_samples_leaf=10,
    random_state=42
)

# EconML API: fit(Y, T, X=...)  where T = treatment (our D)
est.fit(Y_train, W_train, X=X_train)

cate_pred = est.effect(X_test)                      # <-- point estimates
lb, ub = est.effect_interval(X_test, alpha=0.05)    # <-- 95% CI

corr = np.corrcoef(cate_pred, true_test)[0, 1]
coverage = np.mean((lb <= true_test) & (true_test <= ub))

print(f"Mean predicted CATE: ${cate_pred.mean():.0f}")
print(f"Correlation (predicted vs true): {corr:.3f}")
print(f"95% CI coverage: {coverage * 100:.1f}%")

In [ ]:
# Subgroup analysis
analysis = X_test.copy()
analysis['CATE'] = cate_pred
analysis['true_CATE'] = true_test
analysis['income_q'] = pd.qcut(analysis['income'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print("CATE by Income Quartile:")
print(analysis.groupby('income_q', observed=False)[['CATE', 'true_CATE']].mean().round(0))

# Policy targeting
threshold = np.percentile(cate_pred, 50)
target_mask = cate_pred >= threshold
print(f"\nPolicy Targeting (top 50%):")
print(f"  Targeted: ${true_test[target_mask].mean():.0f} vs Random: ${true_test.mean():.0f}")
print(f"  Welfare improvement: {(true_test[target_mask].mean() / true_test.mean() - 1) * 100:.1f}%")

### C.2: Replicating Wager & Athey (2018) Coverage

`CausalForestDML` gives ~91% coverage because DML residualization adds variance not captured by IJ.

`econml.grf.CausalForest` (pure GRF, no DML) matches the paper's ~95% on randomized data.

In [ ]:
from econml.grf import CausalForest                 # <-- pure GRF, no DML

# Wager & Athey (2018) DGP: randomized treatment, no confounding
np.random.seed(42)
n_wa = 5000
X_wa = np.random.uniform(0, 1, (n_wa, 5))

# Their tau function (sigmoid interactions)
tau_wa = ((1 + 1 / (1 + np.exp(-20 * (X_wa[:, 0] - 1/3)))) *
          (1 + 1 / (1 + np.exp(-20 * (X_wa[:, 1] - 1/3)))))
D_wa = np.random.binomial(1, 0.5, n_wa)
Y_wa = tau_wa * D_wa + np.random.normal(0, 1, n_wa)

X_wa_tr, X_wa_te, D_wa_tr, D_wa_te, Y_wa_tr, Y_wa_te = \
    train_test_split(X_wa, D_wa, Y_wa, test_size=0.3, random_state=42)
true_wa_te = ((1 + 1 / (1 + np.exp(-20 * (X_wa_te[:, 0] - 1/3)))) *
              (1 + 1 / (1 + np.exp(-20 * (X_wa_te[:, 1] - 1/3)))))

cf_wa = CausalForest(n_estimators=4000, min_samples_leaf=5, random_state=42)
cf_wa.fit(X_wa_tr, D_wa_tr, Y_wa_tr)               # <-- fit(X, T, Y)

pt_wa, var_wa = cf_wa.predict_and_var(X_wa_te)      # <-- point + variance
pt_wa = pt_wa.flatten()
se_wa = np.sqrt(var_wa.flatten())
lb_wa = pt_wa - 1.96 * se_wa
ub_wa = pt_wa + 1.96 * se_wa
cov_wa = np.mean((lb_wa <= true_wa_te) & (true_wa_te <= ub_wa))

print(f"Wager & Athey DGP (n={n_wa}, randomized):")
print(f"  GRF CausalForest 95% CI coverage: {cov_wa * 100:.1f}%")
print(f"  Correlation: {np.corrcoef(pt_wa, true_wa_te)[0, 1]:.3f}")

## Part D: GRF with Instrumental Variables

When treatment $D_i$ is endogenous, we need an instrument $Z_i$.

**Score**: $\psi = (Z_i - p(x)) \cdot (Y_i - \mu(x) - \theta(x)(D_i - e(x)))$

In [ ]:
np.random.seed(42)
n_iv = 3000

X_iv = np.random.normal(0, 1, (n_iv, 3))
U = np.random.normal(0, 1, n_iv)                    # unobserved confounder
Z_iv = np.random.binomial(1, 0.5, n_iv).astype(float)  # instrument

# Endogenous treatment
W_iv_latent = 0.8 * Z_iv + 0.3 * X_iv[:, 0] + 0.4 * U + np.random.normal(0, 0.3, n_iv)
W_iv = (W_iv_latent > 0.5).astype(float)

# True CATE: tau(x) = 2 + X1
true_tau_iv = 2.0 + X_iv[:, 0]

# Y with confounder interaction: U * (1 + 0.5*X1)
Y_iv = (true_tau_iv * W_iv + 1.0 * X_iv[:, 1] +
        0.8 * U * (1 + 0.5 * X_iv[:, 0]) + np.random.normal(0, 1, n_iv))

print(f"True ATE: {true_tau_iv.mean():.2f}")
print(f"Corr(Z, W) = {np.corrcoef(Z_iv, W_iv)[0, 1]:.3f}")

In [ ]:
from econml.grf import CausalIVForest               # <-- GRF with IV

idx_iv = np.arange(n_iv)
X_iv_tr, X_iv_te, Y_iv_tr, Y_iv_te, W_iv_tr, W_iv_te, \
    Z_iv_tr, Z_iv_te, _, idx_iv_te = \
    train_test_split(X_iv, Y_iv, W_iv, Z_iv, idx_iv, test_size=0.3, random_state=42)
true_tau_iv_test = true_tau_iv[idx_iv_te]

iv_forest = CausalIVForest(n_estimators=1000, min_samples_leaf=10, random_state=42)
iv_forest.fit(X_iv_tr, W_iv_tr, Y_iv_tr, Z=Z_iv_tr)  # <-- fit(X, T, Y, Z=...)

cate_iv_raw = iv_forest.predict(X_iv_te)
cate_iv_pred = (cate_iv_raw.flatten() if hasattr(cate_iv_raw, 'flatten')
                else np.array(cate_iv_raw).flatten())

print(f"IV Forest mean CATE: {cate_iv_pred.mean():.2f} (true: {true_tau_iv_test.mean():.2f})")

In [ ]:
# Compare: naive (ignoring endogeneity) vs IV
naive_forest = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=50, random_state=42),
    model_t=RandomForestRegressor(n_estimators=50, random_state=42),
    n_estimators=500, min_samples_leaf=10, random_state=42)
naive_forest.fit(Y_iv_tr, W_iv_tr, X=X_iv_tr)
cate_naive_pred = naive_forest.effect(X_iv_te)

rmse_iv = np.sqrt(np.mean((cate_iv_pred - true_tau_iv_test) ** 2))
rmse_naive = np.sqrt(np.mean((cate_naive_pred - true_tau_iv_test) ** 2))

print(f"{'':>25} {'Naive':>8} {'IV':>8}")
print(f"{'Mean CATE':>25} {cate_naive_pred.mean():8.2f} {cate_iv_pred.mean():8.2f}")
print(f"{'True mean CATE':>25} {true_tau_iv_test.mean():8.2f} {true_tau_iv_test.mean():8.2f}")
print(f"{'RMSE':>25} {rmse_naive:8.3f} {rmse_iv:8.3f}")
print(f"{'Mean bias':>25} {cate_naive_pred.mean() - true_tau_iv_test.mean():+8.3f} {cate_iv_pred.mean() - true_tau_iv_test.mean():+8.3f}")

# Subgroup analysis
X_iv_df = pd.DataFrame(X_iv_te, columns=['X1', 'X2', 'X3'])
X_iv_df['IV_CATE'] = cate_iv_pred
X_iv_df['Naive_CATE'] = cate_naive_pred
X_iv_df['true_CATE'] = true_tau_iv_test
X_iv_df['X1_group'] = pd.qcut(X_iv_df['X1'], 3, labels=['Low', 'Mid', 'High'])

print("\nCATE by X1 tercile (true tau = 2 + X1):")
print(X_iv_df.groupby('X1_group', observed=False)[['IV_CATE', 'Naive_CATE', 'true_CATE']].mean().round(2))

## Discussion Questions

**Q1**: Why can't we use $\hat{\tau}^2$ directly to select splits?

<details>
<summary>Click to reveal</summary>

$E[\hat{\tau}^2] = \tau^2 + \text{Var}(\hat{\tau})$. Small leaves have high variance, inflating $\hat{\tau}^2$. EMSE subtracts $\hat{V}$ to correct.

</details>

---

**Q2**: What happens without honest estimation?

<details>
<summary>Click to reveal</summary>

The tree selects splits where $\hat{\tau}$ is large partly due to noise. Using the same data to estimate inherits that noise → biased, CIs with ~70% coverage instead of 95%.

</details>

---

**Q3**: Why does DML achieve $\sqrt{n}$-rate despite using slow ML nuisance estimators?

<details>
<summary>Click to reveal</summary>

Neyman orthogonality: first-order nuisance errors vanish. Only the product remains: $n^{-1/4} \times n^{-1/4} = n^{-1/2}$.

</details>

---

**Q4**: When should we use `CausalIVForest` instead of `CausalForestDML`?

<details>
<summary>Click to reveal</summary>

When treatment is endogenous (unobserved confounders). IV forest uses an instrument $Z$ to identify the causal effect. Without IV, confounding biases CATE estimates.

</details>